# MAGNET: African Tokenization Fairness — Complete Colab Training Pipeline

## Overview

Trains and inspects MAGNET (Ahia et al., 2024) on 9 African languages, testing whether per-language boundary predictors beat the paper's script-level design. Mounts Drive, clones the repo, trains the validated β=0.5 baseline, inspects the result, then ends with a documented failed experiment (tuned per-language β collapsed training).

## Prerequisites

- Google Drive with a `DATA_ROOT` folder (see `data_card.md`)
- This repo is public — no GitHub PAT needed
- A GPU runtime: `Runtime > Change runtime type > T4 GPU`

### Why mount Drive

Colab's disk is wiped on disconnect, so the corpora and checkpoints need to live on Drive instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Repo setup

Public repo, so a plain clone with no auth works.

In [ ]:
!git clone https://github.com/mohamad-755/magnet-african-tokenization.git
%cd magnet-african-tokenization

### Installing dependencies

Colab already ships GPU torch; this just adds sentencepiece, pyyaml, tqdm.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# make sure the repo root is importable, and confirm a GPU is attached
import os, sys
REPO_ROOT = os.getcwd()
sys.path.insert(0, REPO_ROOT) if REPO_ROOT not in sys.path else None
os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")

import torch
print("CUDA available:", torch.cuda.is_available())

## Training Setup

Standard hourglass transformer config (d_model=256, 4 heads, batch 16, lr 5e-5), close to the paper's Appendix D.1, for 5000 steps — a reduced budget given time constraints.

β=0.5 for every language isn't a placeholder, it's the only config I found that actually trains stably. A per-language-tuned β (paper Eq. 4) collapsed training instead — see the appendix at the end.

Uses per-language boundary predictors (`LanguageRoutedBoundaryPredictor`), not the paper's script-level ones — that's this project's own extension.

In [ ]:
# training run: per-language routing, uniform beta=0.5 for all 9 languages
# (order matches dataset.LANGUAGES: sw,zu,yo,ig,ha,ny,am,rw,wo)
# re-running overwrites --checkpoint-dir below — change it to keep old runs
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5 \
  --reg-weight 1.0 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language \
  --log-every 50 \
  --eval-every 1000

### What to expect

`lm_loss` should trend down; `reg_loss` should stay bounded rather than blow up or collapse. Logs cycle through all 9 languages round-robin, with an eval block every 1000 steps and checkpoints saved to Drive along the way. ~15-20 min on a T4.

## Inspection & Analysis

Loads the trained checkpoint, prints example segmentations per language, and reports bytes/segment across the full eval set.

In [ ]:
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

## What I found

bytes/segment is just total eval bytes divided by predicted segments — higher means coarser segmentation. What actually matters is whether it's *consistent* across languages, measured by coefficient of variation (CV): lower is fairer.

Giving each language its own predictor didn't help. Isolating the 8 Latin-script languages (Amharic already had a dedicated predictor either way, so it doesn't count), per-language routing's CV came out almost 3x worse than sharing one predictor per script. My read: sharing weights across languages was quietly doing some of the fairness work on its own, and splitting them apart removed that.

MAGNET also isn't script-aware the way BPE is — its raw byte boundaries can split a single Ge'ez character in half, something BPE can't do by construction. Full numbers for both comparisons are in `results/`.

## Troubleshooting

- **Disconnects mid-training**: resume with `--resume .../latest.pt`, keeping `--total-steps`/`--warmup-ratio` identical or the LR schedule won't match.
- **Drive hangs reading Hausa's corpus**: a known Drive issue with that specific 252MB file — re-sync it in Drive if it happens.
- **Clone fails**: check your connection, not GitHub auth — the repo's public.
- **No GPU**: set the runtime to T4 and rerun from the top.
- **`--beta-by-language` errors out**: needs exactly 9 comma-separated values.

---

## Known limitation (documented failure)

These cells reproduce a run that doesn't work, kept for transparency rather than as guidance. Per-language β from Eq. 4 collapsed training to near-zero real boundaries across all 9 languages, regardless of how different their targets were. Lowering `reg_weight` 10x didn't fix it. See `README.md` for more.

In [ ]:
# documented failure — per-language beta (Eq. 4), reg_weight=1.0, collapses
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.1571,0.1069,0.1469,0.1572,0.1730,0.1423,0.0810,0.1381,0.2081 \
  --reg-weight 1.0 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta \
  --log-every 50 \
  --eval-every 1000

In [ ]:
# expect avg_segments/example close to 1.00 for every language
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

In [ ]:
# same beta targets, reg_weight cut 10x (1.0 -> 0.1) — still collapses
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.1571,0.1069,0.1469,0.1572,0.1730,0.1423,0.0810,0.1381,0.2081 \
  --reg-weight 0.1 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta_lowreg \
  --log-every 50 \
  --eval-every 1000

In [ ]:
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta_lowreg/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

### What I take from this

Both attempts landed on identical eval stats, which makes sense once you realize a fully collapsed predictor's output stops depending on the model at all — it's just data length divided by example count. Cutting `reg_weight` 10x clearly wasn't the fix. With more time I'd try warming up the regularizer instead of applying it at full strength from step 0.